# Getting each street segment's crime score

#### This notebook contains code to obtain the crime density-based scores for day and night.

#### DO NOT run this notebook. This notebook should only be run by STARSWalkability/safety.scoring.ipynb, in combination with the crime buffer and light density scores to generate an overall safety score.

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
from shapely.geometry import box, Point
import osmnx as ox
from sklearn.neighbors import KernelDensity
from shapely import wkt
from shapely.geometry import LineString, MultiLineString
import math
import os

# boundaries
north = 47.62425802431359
south = 47.596252480947555
east  = -122.31765250756945
west  = -122.35919327219091

EARTH_RADIUS_M = 6_371_000.0

bbox = (west, south, east, north)

# graph = ox.graph_from_bbox(
#     bbox,
#     network_type='walk',
# )


day_crimes = pd.read_csv('../crime_density/data/daytime_crimes_alltypes_weighted.csv')
night_crimes = pd.read_csv('../crime_density/data/nighttime_crimes_alltypes_weighted.csv')
edges_df = pd.read_csv('../routing/all_scored_edges_filtered_with_ai.csv')
if edges_df["geometry"].dtype == object:
    edges_df["geometry"] = edges_df["geometry"].apply(wkt.loads)


/var/folders/p7/5hrm9t_12rs2c4jgh750nc200000gn/T/ipykernel_32872/4085612112.py:32: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  edges_df = pd.read_csv('../routing/all_scored_edges_filtered_with_ai.csv')


In [14]:
edges_footways = edges_df[edges_df["highway"] == "footway"]

In [15]:
print(edges_footways.describe())

       Unnamed: 0.3  Unnamed: 0.2  Unnamed: 0.1    Unnamed: 0             u  \
count  18972.000000  18972.000000  18972.000000  18972.000000  1.897200e+04   
mean    9485.500000   9485.500000   9485.500000   9485.500000  8.451910e+09   
std     5476.888989   5476.888989   5476.888989   5476.888989  2.608450e+09   
min        0.000000      0.000000      0.000000      0.000000  2.993814e+07   
25%     4742.750000   4742.750000   4742.750000   4742.750000  5.772005e+09   
50%     9485.500000   9485.500000   9485.500000   9485.500000  8.836452e+09   
75%    14228.250000  14228.250000  14228.250000  14228.250000  1.074765e+10   
max    18971.000000  18971.000000  18971.000000  18971.000000  1.299323e+10   

                  v           key        length  lanes  service  ...  \
count  1.897200e+04  18972.000000  18972.000000    8.0      0.0  ...   
mean   8.451910e+09      0.004269     27.783977    1.0      NaN  ...   
std    2.608450e+09      0.066800     29.047594    0.0      NaN  ...   


In [17]:
import numpy as np

# lat/lon in radians
lats = np.deg2rad(day_crimes['Latitude'].to_numpy())
lons = np.deg2rad(day_crimes['Longitude'].to_numpy())

# center
phi = lats.mean()
lam = lons.mean()

# angular distance because we are using haversine/great-circle distance
dlat = lats - phi
dlon = lons - lam
a = np.sin(dlat/2)**2 + np.cos(phi) * np.cos(lats) * np.sin(dlon/2)**2
dist_rad = 2 * np.arcsin(np.sqrt(a))


# of crime pts
n = dist_rad.size
print(n)

#std deviation
s = dist_rad.std(ddof=1)
iqr = np.percentile(dist_rad, 75) - np.percentile(dist_rad, 25)

# bandwidth in radians based on Silverman's
h_rad = 0.9 * min(s, iqr/1.34) * (n ** (-1/5))


# convert to meters
bandwidth_m = h_rad * EARTH_RADIUS_M

print(h_rad)
print(bandwidth_m)

6847
1.1231161133923236e-05
71.55372758422493


## Configure/Organize

In [ ]:
# Add sigmas to the list, each sigma value will generate a df with scores for day/night with that sigma val
# dont include / after a folder name
# file name will be 'cds_{sigma value}.csv'

# sigmas = [0.01, 0.1, 1, 10, 20, 30, 40, 50, 67, 75, 100]
sigmas = [67]
path = 'density_scores'
num_times = len(sigmas)



NameError: name 'sigmas' is not defined

1. Uses: 
    - array of the lat/lon of all daytime crimes in past year
    - array of lat/lon of all nighttime crimes in past year
    - corresponding arrays of normalized weights for all crimes in day/night

2. Normalized Weight column is obtained by:
    1. First get recency of each crime: 
        - get how many days ago it happened ( now - Offense DateTime)
        - use exponential decay function (e^ (-lam * daysAgo)), which gives val between 0 and 1, 1 most recent
            - because its like smooth fade, matters less over time
        - multiply by Crime Weight
        - Normalize by getting min and max, and converting all of them to be between 0 and 1
    
3. Use scikit learn's KernelDensity to apply Gaussian Kernel Density Estimation
    - each point contributes a gaussian kernel (put a smooth bump at that point on map)
        - each point corresponds to a probability density function
    - you ask it for the sample/estimated density score at a point via kde.score_samples()
        - it will evaluate the sum at that point location (sums up the bumps)
    - returns logarithm of estimated density(s)
        - can ask it and receive log densities for multiple values at once
    - to get the raw density, have to exponent it
    - the raw density is the height of the estimated PDF, the value obtained at a point measures the relative intensity of crimes nearby
    - ex. at a point the KDE is 0.98 out of 1: that can mean multiple things - lots of recent, low-severity crimes, middling amount of semi-recent high-severity crimes, etc.
        - "intensity" = measures if crimes severity, recency, and count


In [15]:
def fit_gaussian_haversine_kde(bandwidth, coords_radians, weights):
    kde = KernelDensity(bandwidth=bandwidth, metric='haversine', kernel='gaussian', algorithm='ball_tree')
    kde.fit(coords_radians, sample_weight=weights)
    return kde

def normalize_densities(density_list):
    arr = np.array(density_list, dtype=float)
    dmin, dmax = arr.min(), arr.max()
    if dmax == dmin:
        return np.zeros_like(arr)
    norm_arr = (arr - dmin) / (dmax - dmin)
    return norm_arr

def get_raw_densities(edges_df, kde):
    all_raw_densities = []
    for idx, row in edges_df.iterrows():
        geom = row["geometry"]

        # get all the coords in the Linestring
        if geom.geom_type == "LineString":
            coords = list(geom.coords) 
        elif geom.geom_type == "MultiLineString":
            coords = []
            for g in geom.geoms:
                coords.extend(list(g.coords))
        else:
            coords = []

        if coords:
            # convert to numpy arr
            # coords: (m, 2) in (lon_deg, lat_deg)
            coords = np.array(coords)
            lons_deg = coords[:, 0]
            lats_deg = coords[:, 1]

            # reorder + convert
            lats_rad = np.deg2rad(lats_deg)
            lons_rad = np.deg2rad(lons_deg)
            query_points = np.column_stack([lats_rad, lons_rad])

            # score
            log_densities = kde.score_samples(query_points)
            raw_densities = np.exp(log_densities)

            # average along the edge
            avg_density = raw_densities.mean()
        else:
            avg_density = 0.0

        all_raw_densities.append(avg_density)
    return all_raw_densities


In [16]:


def compute_density_crime_score(crime_coords_df, density_radius_m, edges_df):
    edges_mapped_df = edges_df.copy()
    # extract coordinates, weights
    coords_radians = (crime_coords_df[['Latitude', 'Longitude']].values * (math.pi/180))
    weights = crime_coords_df['Crime Weight'].values

    # bandwidth
    bandwidth = density_radius_m / EARTH_RADIUS_M

    kde = fit_gaussian_haversine_kde(bandwidth, coords_radians, weights)

    raw_densities = get_raw_densities(edges_df, kde)

    edges_mapped_df = edges_df.copy()
    edges_mapped_df['RawAvgDensity'] = raw_densities
    # edges_mapped_df['NormalizedAvgDensity'] = normalize_densities(edges_mapped_df['RawAvgDensity'].values)
    # edges_mapped_df['CrimeScore'] = (1-edges_mapped_df['NormalizedAvgDensity'])
    
    return edges_mapped_df

def compute_density_crime_score_to_csv(crime_coords_df, density_radius_m, edges_df, path_filename):
    edges_mapped_df = compute_density_crime_score(crime_coords_df, density_radius_m, edges_df)
    edges_mapped_df.to_csv(f'{path_filename}.csv')
    return


In [17]:
# function to be called

def cds_to_df(edges_df, sigma):
    day_cd_crimes = pd.read_csv('../crime_density/data/day_crimes_weighted.csv')
    night_cd_crimes = pd.read_csv('../crime_density/data/night_crimes_weighted.csv')

    d = compute_density_crime_score(day_cd_crimes, sigma, edges_df)
    n = compute_density_crime_score(night_cd_crimes, sigma, edges_df)
    raw_day = d['RawAvgDensity'].to_numpy()
    raw_night = n['RawAvgDensity'].to_numpy()
    global_raw = np.concatenate([raw_day, raw_night])

    gmin = np.nanmin(global_raw)
    gmax = np.nanmax(global_raw)
    denom = gmax - gmin

    if denom <= 0: 
        d['NormGlobal'] = 0.0
        n['NormGlobal'] = 0.0
    else:
        d['NormGlobal'] = (raw_day  - gmin) / denom
        n['NormGlobal'] = (raw_night - gmin) / denom


    d['DayDensityCrimeScore'] = 1 - d['NormGlobal']
    n['NightDensityCrimeScore'] = 1 - n['NormGlobal']

    edges_df['safety_score_day_density'] = ((0.9*d['DayDensityCrimeScore']) + 0.1)
    edges_df['safety_score_night_density'] = ((0.9*n['NightDensityCrimeScore']) + 0.1)

    edges_df['ai_density_day_average'] = (edges_df['safety_score_day_density'] + edges_df['AI_avg_safety_score'])/2
    edges_df['ai_density_night_average'] = (edges_df['safety_score_night_density'] + edges_df['AI_avg_safety_score'])/2


    return edges_df

In [21]:
for sigma in sigmas:
    df = cds_to_df(edges_df, sigma)
    df.to_csv(f'density_scores/cds_{sigma}.csv')

In [15]:
df.describe()

,u,v,key,length,density_day,density_night
count,2.588600e+04,2.588600e+04,25886.000000,25886.000000,25886.000000,25886.000000
mean,8.282471e+09,8.282471e+09,0.009349,27.697297,0.886314,0.885652
std,2.988861e+09,2.988861e+09,0.103211,29.923901,0.090376,0.089644
min,2.993814e+07,2.993814e+07,0.000000,0.239832,0.345058,0.100000
25%,5.767399e+09,5.767399e+09,0.000000,7.884306,0.862786,0.856438
50%,8.920207e+09,8.920207e+09,0.000000,17.274127,0.908800,0.908202
75%,1.075794e+10,1.075794e+10,0.000000,37.951390,0.941545,0.941400
max,1.299323e+10,1.299323e+10,3.000000,523.929678,1.000000,1.000000
